### Data Ingestion


In [1]:
###Document Structure
from pathlib import Path

from langchain_community.document_loaders import (
    TextLoader,
    PyPDFLoader,
    Docx2txtLoader,
    CSVLoader,
    UnstructuredPowerPointLoader,
    UnstructuredExcelLoader
)

def load_file(file_path):
    """
    Load a file using the appropriate loader based on its extension.
    """

    path = Path(file_path)
    extension = path.suffix.lower()

    # Markdown / Text
    if extension in [".md", ".txt"]:
        loader = TextLoader(
            str(path),
            encoding="utf-8"
        )

    # PDF
    elif extension == ".pdf":
        loader = PyPDFLoader(str(path))

    # Word
    elif extension == ".docx":
        loader = Docx2txtLoader(str(path))

    # CSV
    elif extension == ".csv":
        loader = CSVLoader(str(path))

    # PowerPoint
    elif extension == ".pptx":
        loader = UnstructuredPowerPointLoader(str(path))

    # Excel
    elif extension in [".xlsx", ".xls"]:
        loader = UnstructuredExcelLoader(str(path))

    else:
        print(f"Skipping unsupported file: {path}")
        return []

    try:
        return loader.load()

    except Exception as e:
        print(f"Failed to load {path}: {e}")
        return []


def load_knowledge_base(directory="../knowledge"):
    """
    Load all supported files recursively from the knowledge directory.
    """

    documents = []

    knowledge_path = Path(directory)

    supported_extensions = {
        ".md",
        ".txt",
        ".pdf",
        ".docx",
        ".csv",
        ".pptx",
        ".xlsx",
        ".xls"
    }

    for file_path in knowledge_path.rglob("*"):

        if not file_path.is_file():
            continue

        if file_path.suffix.lower() not in supported_extensions:
            continue

        file_documents = load_file(file_path)
        documents.extend(file_documents)

    return documents

# Load entire knowledge base
documents = load_knowledge_base("../knowledge")

C:\Users\abhis\AppData\Local\Temp\ipykernel_11312\3107557812.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (
c:\Users\abhis\Desktop\Project\Portfolio RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = text_splitter.split_documents(documents)

print(f"Original documents: {len(documents)}")
print(f"Total chunks: {len(chunks)}")

texts = [chunk.page_content for chunk in chunks]

Original documents: 13
Total chunks: 82


### Embedding and VectorStoreDB

In [3]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
class EmbeddingManager:
    """Handles document embedding generation using Sentence Transformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager.

        Args:
            model_name: HuggingFace model name for sentence transformer.
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the Sentence Transformer."""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Embedding model loaded successfully. Dimendions : {self.model.get_embedding_dimension()}")
        except Exception as error:
            raise RuntimeError(
                f"Failed to load embedding model '{self.model_name}'."
            ) from error

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for list of texts

        Args:
            texts : List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts..")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape : {embeddings.shape}")
        return embeddings

# initialize the embedding manager
embedding_manager = EmbeddingManager()
embeddings = embedding_manager.generate_embeddings(texts)

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10613.80it/s]


Embedding model loaded successfully. Dimendions : 384
Generating embeddings for 82 texts..


Batches: 100%|██████████| 3/3 [00:01<00:00,  2.19it/s]

Generated embeddings with shape : (82, 384)


### VectorStore

In [5]:
import os
class VectorStore:
    """
    Manages document embeddings in a ChromaDB vector store 
    """
    def __init__(
            self,
            collection_name : str="portfolio_knowledge",
            persist_directory: str="../knowledge/vector_store"
    ):
        """ 
        Initialize the Vector Store

        Args:
            collection_name : Name of the chromaDB collection
            persist_directory : Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection."""
        try:
            # Create persistent ChromaDB directory
            os.makedirs(
                self.persist_directory,
                exist_ok=True
            )
            # Create persistent ChromaDB client
            self.client = chromadb.PersistentClient(
                path=self.persist_directory
            )
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "Portfolio knowledge embeddings for RAG",
                    "hnsw:space": "cosine"
                }
            )
            print(
                f"Vector store initialized. "
                f"Collection: {self.collection_name}"
            )

            print(
                f"Existing documents in collection: "
                f"{self.collection.count()}"
            )
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(
        self,
        documents: List[Any],
        embeddings: np.ndarray
    ):
        """
        Add documents and their embeddings to the vector store.

        Args:
            documents: List of LangChain documents.
            embeddings: Corresponding embeddings for the documents.
        """
        # Make sure the number of documents matches
        # the number of embeddings
        if len(documents) != len(embeddings):
            raise ValueError(
                "Number of documents must match number of embeddings"
            )
        print(
            f"Adding {len(documents)} documents to vector store..."
        )
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        # Process each document and its corresponding embedding
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            source = doc.metadata.get("source", "unknown")
            doc_id = f"{source}_{i}".replace("\\", "_").replace("/", "_")
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)

            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add documents to ChromaDB collection
        try:
            self.collection.upsert(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(
                f"Successfully added {len(documents)} "
                f"documents to vector store"
            )
            print(
                f"Total documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:
            print(
                f"Error adding documents to vector store: {e}"
            )
            raise
vectorstore = VectorStore()

Vector store initialized. Collection: portfolio_knowledge
Existing documents in collection: 0


In [6]:
# convert the chunks to embeddings
texts = [chunk.page_content for chunk in chunks]

# generate the embeddings
embeddings = embedding_manager.generate_embeddings(texts)

# store in the vector database
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 82 texts..


Batches: 100%|██████████| 3/3 [00:01<00:00,  2.07it/s]


Generated embeddings with shape : (82, 384)
Adding 82 documents to vector store...
Successfully added 82 documents to vector store
Total documents in collection: 82


### Retriever Pipeline from VectorStore

In [7]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever.

        Args:
            vector_store: Vector store containing document embeddings.
            embedding_manager: Manager for generating query embeddings.
        """

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query.

        Args:
            query: The search query.
            top_k: Number of top results to return.
            score_threshold: Minimum similarity score threshold.

        Returns:
            List of dictionaries containing retrieved documents and metadata.
        """

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, "f"Score threshold: {score_threshold}")

        try:
            # Generate query embedding
            query_embedding = (self.embedding_manager.generate_embeddings([query])[0])

            # Search in vector store
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_docs = []
            if results['documents'] and results['documents'][0]:

                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(
                    zip(ids, documents, metadatas, distances)
                ):
                    # Convert distance to similarity score
                    # ChromaDB uses cosine distance
                    similarity_score = 1 - distance

                    # Apply similarity threshold
                    if similarity_score >= score_threshold:

                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })

                print( f"Retrieved {len(retrieved_docs)} " f"documents (after filtering)")
            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(
                f"Error during retrieval: {e}"
            )
            return []
# instantiate using the existing notebook variables
rag_retriever = RAGRetriever(vectorstore, embedding_manager)


### Integration VectorDB context pipeline with LLM

In [12]:
### RAG pipeline with LLM
from urllib import response

from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("API_KEY")
llm = ChatGoogleGenerativeAI(
    api_key = api_key,
    model="gemini-3.6-flash",
    max_tokens=1024
)

# RAG Function : retrieve context + generate response
def rag_simple(query, retriever, llm, top_k=5):
    """
    Perform a simple RAG (Retrieval-Augmented Generation) operation.

    Args:
        query: The input query string.
        retriever: An instance of RAGRetriever for document retrieval.
        llm: A language model instance for generating responses.
        top_k: Number of top documents to retrieve.

    Returns:
        A dictionary containing the generated response and retrieved documents.
    """
    # Retrieve relevant documents
    results = retriever.retrieve(query, top_k=top_k)

    # Prepare context for LLM
    context = "\n\n".join([doc['content'] for doc in results]) if results else "No relevant documents found."
    # Create prompt
    prompt = f"""Context:{context} Question: {query}Answer:"""

    # Generate response using LLM
    response = llm.invoke(prompt)
    response = llm.invoke(prompt)

    if isinstance(response.content, str):
        return response.content

    return "\n".join(
        block["text"]
        for block in response.content
        if block.get("type") == "text"
    )

In [14]:
answer = rag_simple("Does Abhishek know Docker?", rag_retriever, llm, top_k=5)
print("Answer:", answer)

Retrieving documents for query: 'Does Abhishek know Docker?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts..


Batches: 100%|██████████| 1/1 [00:00<00:00, 126.41it/s]

Generated embeddings with shape : (1, 384)
Retrieved 5 documents (after filtering)


Answer: Based on the provided context, there is no mention of Docker in Abhishek Yadav's skills, projects, or technical experience.
